# Tissue-niche annotation with TissueAgent, Biomni and SpatialAgent

Compare named tissue annotations on developing heart MERFISH and SpatialFusion Xenium OVCA. All methods receive the same stripped section inputs and label vocabulary; the default model is GPT-5.1. LLMiniST is deferred. See [the guide](../docs/tissue_niche_agent_baselines.md) for environments and process-isolation limits.

Download OVCA first with `python -m demo.tissue_niche.spatialfusion_ovca download`. Model calls occur only in the explicitly marked execution cell.

In [ ]:
from pathlib import Path
import sys
import yaml

ROOT = Path.cwd()
if not (ROOT / 'demo/tissue_niche').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from demo.tissue_niche.agent_inputs import prepare_dataset
from demo.tissue_niche.agent_baselines import preflight
from demo.tissue_niche.run_agents import run_study
from demo.tissue_niche.evaluation import evaluate_study
from demo.tissue_niche.plot_agent_comparison import plot_performance, plot_spatial

config = yaml.safe_load((ROOT / 'demo/tissue_niche/configs/agent_comparison.yaml').read_text())
study_dir = ROOT / 'demo/outputs/tissue_niche/agent_comparisons/tissue-niche-agents-gpt51-notebook-v3'
config

Prepare all sections with expression, cell types and coordinates only. Private truth remains in the preparation directory and is read by the evaluator. Preparation and preflight do not call an LLM.

In [ ]:
prepared = {dataset: prepare_dataset(dataset) for dataset in config['datasets']}
{dataset: {'cells': data['n_cells'], 'scored': data['n_scored'], 'sections': len(data['sections'])}
 for dataset, data in prepared.items()}

In [ ]:
{method: preflight(method, config) for method in config['methods']}

The next cell launches the full configured study and incurs provider usage. Use a new study directory for each new experiment. To run only the new baselines, set `config['methods'] = ['biomni', 'spatialagent']` before execution. Configurable models and requested seeds are recorded per run. Finalized failures are retained; an identical interrupted study can resume with `resume=True`.

In [ ]:
study = run_study(config, study_dir)
[(run['dataset'], run['method'], run['seed'], run['section_id'], run['status'])
 for run in study['runs']]

Evaluate returned labels directly, with missing/unknown predictions counted as Unmatched. No LLM relabeling is performed. Macro F1 and balanced accuracy are pooled across sections within each dataset; unassigned OVCA truth is excluded from scoring while retained as input context.

In [ ]:
metrics = evaluate_study(study_dir)
metrics

In [ ]:
figures = plot_performance(study_dir, study_dir / 'figures')
figures += plot_spatial(study_dir, study_dir / 'figures')
from IPython.display import Image, display
for path in figures:
    if path.suffix == '.png':
        display(Image(filename=str(path)))